# Redis 缓存与 Celery 异步任务实战

> 本 Notebook 展示了如何利用 Redis 实现简单的分布式锁以及幂等性校验逻辑。

## 1. 模拟 Redis 锁
演示 SET NX (Set if Not Exists) 的原子性思想。

In [ ]:
# 模拟 Redis 客户端行为
class FakeRedis:
    def __init__(self): self.db = {}
    def set(self, k, v, nx=False):
        if nx and k in self.db: return False
        self.db[k] = v
        return True

r = FakeRedis()
print(f"First attempt: {r.set('lock:1', 'val', nx=True)}")
print(f"Second attempt: {r.set('lock:1', 'val', nx=True)}")

## 2. 任务幂等性设计
在处理异步任务前，先检查全局标记位。

In [ ]:
processed_ids = set()

def run_task(task_id):
    if task_id in processed_ids:
        print(f"Skip: Task {task_id} already done")
        return
    print(f"Running Task {task_id}...")
    processed_ids.add(task_id)

run_task(101)
run_task(101)